# Domain Shift Variance Analysis

This notebook analyzes which layers in the Mamba-Vision backbone experience the highest domain shift between BDD-Day and BDD-Night (or ACDC). This helps identify which layers should be targeted for LoRA adapter injection.

In [2]:
from __future__ import annotations

import sys
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import json
import numpy as np
import torchvision.transforms as transforms

import random

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

RUN_MODE = 'pilot'  # pilot | full
# PREPARE_DATA = False
# AUTO_CONFIRM_FULL = False

print('Repo root:', REPO_ROOT)
print('Run mode:', RUN_MODE)


Repo root: /teamspace/studios/this_studio/Hot-Peppers-Company-Computer-Vision
Run mode: pilot


In [3]:
CONFIG_PATH = REPO_ROOT / 'configs/training/coco_base.yaml'
CONFIG_PATH

PosixPath('/teamspace/studios/this_studio/Hot-Peppers-Company-Computer-Vision/configs/training/coco_base.yaml')

In [4]:
from pipelines.dependencies import check_packages, assert_required_packages

required = ['torch', 'torchvision', 'yaml', 'safetensors', 'tqdm', 'einops']
optional = ['fiftyone', 'mambavision', 'wandb']
status = check_packages(required + optional)
status

{'torch': True,
 'torchvision': True,
 'yaml': True,
 'safetensors': True,
 'tqdm': True,
 'einops': True,
 'fiftyone': True,
 'mambavision': True,
 'wandb': True}

In [5]:
assert_required_packages(['torch', 'torchvision', 'yaml', 'safetensors', 'einops'])
print('Core dependencies look good')

Core dependencies look good


In [6]:
from pipelines.contracts import TrainConfig

cfg = TrainConfig.from_yaml(CONFIG_PATH)
cfg.model.model_file = str((REPO_ROOT / cfg.model.model_file).resolve())
cfg.ckpt.output_path = str((REPO_ROOT / cfg.ckpt.output_path).resolve())
if cfg.model.base_checkpoint:
    cfg.model.base_checkpoint = str((REPO_ROOT / cfg.model.base_checkpoint).resolve())
if cfg.data.get('lora_output_path'):
    cfg.data['lora_output_path'] = str((REPO_ROOT / cfg.data['lora_output_path']).resolve())
cfg

TrainConfig(run_name='coco_base', data={'source': 'fiftyone_zoo', 'zoo_name': 'coco-2017', 'split_train': 'train', 'split_val': 'validation', 'manifest_train': 'configs/manifests/coco_train.json', 'manifest_val': 'configs/manifests/coco_val.json', 'export_train_dir': 'data/exports/coco/train', 'export_val_dir': 'data/exports/coco/val', 'max_samples_train': None, 'max_samples_val': None}, model=ModelSection(backbone='mamba_vision_T2', num_classes=8, pretrained=True, checkpoint_path='mambavision_tiny2_1k.pth.tar', base_checkpoint='', model_file='/teamspace/studios/this_studio/Hot-Peppers-Company-Computer-Vision/mamba-vision-ours/model.py'), train=TrainSection(epochs=30, batch_size=8, num_workers=4, lr=0.0001, weight_decay=0.0001, scheduler='cosine', pilot_steps=5, pilot_val_steps=2, precision='fp16', device='cuda', image_size=640, grad_clip_norm=1.0), ckpt=CkptConfig(output_path='/teamspace/studios/this_studio/Hot-Peppers-Company-Computer-Vision/checkpoints/base/coco_base.ckpt', save_top

In [7]:
from pipelines.model_loader import create_model_from_config
from pipelines.training import load_checkpoint, resolve_device

device = resolve_device(cfg.train.device)
model = create_model_from_config(cfg.model, device=str(device))

if cfg.model.base_checkpoint:
    load_checkpoint(REPO_ROOT / cfg.model.base_checkpoint, model)
    print('Loaded base checkpoint:', REPO_ROOT / cfg.model.base_checkpoint)

model = model.to(device)

model.eval()

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


The model and loaded state dict do not match exactly

unexpected key in source state_dict: head.weight, head.bias

Loaded pretrained backbone tensors from /tmp/mamba_vision_T2.pth.tar: 8
Missing keys: ['patch_embed.conv_down.0.weight', 'patch_embed.conv_down.1.weight', 'patch_embed.conv_down.1.bias', 'patch_embed.conv_down.1.running_mean', 'patch_embed.conv_down.1.running_var', 'patch_embed.conv_down.3.weight', 'patch_embed.conv_down.4.weight', 'patch_embed.conv_down.4.bias', 'patch_embed.conv_down.4.running_mean', 'patch_embed.conv_down.4.running_var', 'levels.0.blocks.0.conv1.weight', 'levels.0.blocks.0.conv1.bias', 'levels.0.blocks.0.norm1.weight', 'levels.0.blocks.0.norm1.bias', 'levels.0.blocks.0.norm1.running_mean', 'levels.0.blocks.0.norm1.running_var', 'levels.0.blocks.0.conv2.weight', 'levels.0.blocks.0.conv2.bias', 'levels.0.blocks.0.norm2.weight', 'levels.0.blocks.0.norm2.bias', 'levels.0.blocks.0.norm2.running_mean', 'levels.0.blocks.0.norm2.running_var', 'levels.0.downsamp

MambaVisionOurs(
  (backbone): MambaVision(
    (patch_embed): PatchEmbed(
      (proj): Identity()
      (conv_down): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=0.0001, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(32, 80, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (4): BatchNorm2d(80, eps=0.0001, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU()
      )
    )
    (levels): ModuleList(
      (0): MambaVisionLayer(
        (blocks): ModuleList(
          (0): ConvBlock(
            (conv1): Conv2d(80, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm1): BatchNorm2d(80, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (act1): GELU(approximate='tanh')
            (conv2): Conv2d(80, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (no

In [8]:
class VarianceHook:
    def __init__(self):
        self.variances = {}
        self.handles = []

    def hook_fn(self, name):
        def fn(module, input, output):
            if isinstance(output, torch.Tensor):
                batch_size = output.shape[0]
                if len(output.shape) > 1:
                    flattened = output.view(batch_size, -1)
                    var = torch.var(flattened, dim=0).mean().item()
                    self.variances[name] = var
        return fn

    def register_hooks(self, model):
        for name, module in model.backbone.named_modules():
            if isinstance(module, nn.Linear):
                handle = module.register_forward_hook(self.hook_fn(name))
                self.handles.append(handle)

    def clear(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []
        self.variances = {}

hook_obj = VarianceHook()
hook_obj.register_hooks(model)
print(f"Registered hooks on {len(hook_obj.handles)} linear layers in backbone")

Registered hooks on 76 linear layers in backbone


In [9]:
from pipelines.coco_dataset import build_dataloader
from pipelines.contracts import DatasetManifest

print("Loading BDD Day and Night datasets using official pipeline...")

# Load the manifests generated by Ivan's ninja export script
day_manifest_path = REPO_ROOT / "configs/manifests/bdd_day_train.json"
night_manifest_path = REPO_ROOT / "configs/manifests/bdd_night_train.json"

if not day_manifest_path.exists() or not night_manifest_path.exists():
    print("ERROR: Manifests not found! Did you run export_bdd_ninja_to_coco.py?")
else:
    day_manifest = DatasetManifest.from_json(day_manifest_path)
    night_manifest = DatasetManifest.from_json(night_manifest_path)

    # Build the official dataloaders, but limit to just 32 samples for speed
    day_loader = build_dataloader(
        day_manifest,
        image_size=640,
        batch_size=16,
        num_workers=4,
        shuffle=True,
        max_samples=256  # Only need 2 batches of 16 for variance calculation
    )

    night_loader = build_dataloader(
        night_manifest,
        image_size=640,
        batch_size=16,
        num_workers=4,
        shuffle=True,
        max_samples=256
    )

    print(f"Loaded BDD Day: {len(day_loader.dataset)} images")
    print(f"Loaded BDD Night: {len(night_loader.dataset)} images")


Loading BDD Day and Night datasets using official pipeline...
Loaded BDD Day: 256 images
Loaded BDD Night: 208 images


In [10]:
def compute_variances(loader, model, hook_obj, domain_name):
    hook_obj.clear()
    hook_obj.register_hooks(model)
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):

            images = batch[0] if isinstance(batch, (tuple, list)) else batch
            images = images.to(device)
            
            # Forward pass
            model.backbone.forward_features(images)
            print(f"  Processed batch {batch_idx + 1}")
            
    variances = hook_obj.variances.copy()
    hook_obj.clear()
    print(f"{domain_name} - Collected variances from {len(variances)} layers")
    return variances

if 'day_loader' in locals() and 'night_loader' in locals():
    print("Computing variance for BDD Day...")
    day_variances = compute_variances(day_loader, model, hook_obj, "BDD Day")
    
    print("\nComputing variance for BDD Night...")
    night_variances = compute_variances(night_loader, model, hook_obj, "BDD Night")
    
    # Calculate difference
    variance_diffs = {}
    for layer_name in day_variances.keys():
        if layer_name in night_variances:
            diff = abs(day_variances[layer_name] - night_variances[layer_name])
            variance_diffs[layer_name] = diff
            
    # Sort layers by highest variance shift
    sorted_diffs = sorted(variance_diffs.items(), key=lambda x: x[1], reverse=True)
    print(f"\nSuccessfully calculated variance shifts across {len(sorted_diffs)} layers!")
else:
    print("chedck cfg for correct locals")


Computing variance for BDD Day...
  Processed batch 1
  Processed batch 2
  Processed batch 3
  Processed batch 4
  Processed batch 5
  Processed batch 6
  Processed batch 7
  Processed batch 8
  Processed batch 9
  Processed batch 10
  Processed batch 11
  Processed batch 12
  Processed batch 13
  Processed batch 14
  Processed batch 15
  Processed batch 16
BDD Day - Collected variances from 76 layers

Computing variance for BDD Night...
  Processed batch 1
  Processed batch 2
  Processed batch 3
  Processed batch 4
  Processed batch 5
  Processed batch 6
  Processed batch 7
  Processed batch 8
  Processed batch 9
  Processed batch 10
  Processed batch 11
  Processed batch 12
  Processed batch 13
BDD Night - Collected variances from 76 layers

Successfully calculated variance shifts across 76 layers!


In [12]:
if variance_diffs:
    print("\n" + "="*80)
    print("TOP 10 LAYERS WITH HIGHEST DOMAIN SHIFT (BDD Day vs BDD Night)")
    print("="*80)
    print(f"{'Rank':<6} {'Layer Name':<50} {'Variance Diff':<15}")
    print("-"*80)
    
    for rank, (layer_name, diff) in enumerate(sorted_diffs[:10], start=1):
        print(f"{rank:<6} {layer_name:<50} {diff:.6f}")
    
    print("="*80)
    print("\nThese layers experience the highest domain shift => good candidates for LoRA injection")
else:
    print("No variance differences to display. Check dataset loading.")


TOP 10 LAYERS WITH HIGHEST DOMAIN SHIFT (BDD Day vs BDD Night)
Rank   Layer Name                                         Variance Diff  
--------------------------------------------------------------------------------
1      levels.3.blocks.1.mlp.fc1                          48.411880
2      levels.3.blocks.2.mlp.fc1                          29.001846
3      levels.3.blocks.0.mlp.fc1                          25.925720
4      levels.3.blocks.3.mlp.fc1                          25.498711
5      levels.3.blocks.0.mixer.in_proj                    5.877262
6      levels.3.blocks.1.mixer.in_proj                    4.862558
7      levels.2.blocks.0.mixer.in_proj                    4.313528
8      levels.3.blocks.0.mlp.fc2                          3.489685
9      levels.3.blocks.1.mlp.fc2                          3.358826
10     levels.3.blocks.3.mlp.fc2                          3.066318

These layers experience the highest domain shift => good candidates for LoRA injection
